[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/20.%20The%20SLAP%20for%20Out-of-Gauge%20%28OOG%29%20Containers%20Problem/P20-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 20. The SLAP for Out-of-Gauge (OOG) Containers Problem

## Tier 1: The Pen-and-Paper Method (Mathematical Formulation)

### Purpose and scope

This notebook assigns each out-of-gauge (OOG) container to one available vessel slot. It uses the Hungarian algorithm to minimize the sum of the assignment costs, then checks the resulting assignment against slot and bay restrictions. Stability and lashing scores are also calculated and can be examined in a separate assignment run.

The distinction between optimization and checking matters: the Hungarian algorithm used here solves the cost-based assignment problem, but it does not enforce the weight, height, or bay-weight restrictions. Those restrictions are audited after the assignment. Therefore, a feasible output is a cost-optimal assignment for the basic assignment problem that also passes the audit; this procedure does not, in general, guarantee the least-cost assignment among all assignments satisfying the side constraints.

### Modeling assumptions

- Each container must be assigned to exactly one slot, and each slot can receive at most one container.
- A slot is eligible for a container only if its weight capacity and height limit are sufficient. These conditions are checked after the Hungarian assignment in this tier.
- The total weight assigned to each bay must not exceed that bay's stated limit. This coupled restriction is also checked after assignment.
- Assignment costs, stability scores, lashing coefficients, slot capacities, and bay limits are known and fixed for each run.
- Costs are measured in abstract cost units; container and slot weights are in tons, heights and crane distances are in meters. Stability scores and lashing coefficients are dimensionless in this example.
- The example does not model container-slot compatibility beyond weight and height, spatial interactions within a bay, uncertainty, or changes during operations.

### Solution sequence

1. Represent containers, slots, costs, stability scores, and bay limits as data.
2. Use the Hungarian algorithm to find a minimum-cost one-to-one assignment for the rectangular cost matrix.
3. Audit the assignment for container coverage, slot capacity, individual weight and height limits, and bay-weight limits.
4. Calculate cost, stability, capacity utilization, lashing complexity, and bay-weight distribution.
5. Use the supplied what-if scenarios to examine how changed costs, capacities, or objectives affect the assignment and its feasibility.

### What to examine

- Which container-slot pairs the cost-minimizing assignment selects.
- Whether every audited restriction is satisfied; an assignment that fails any restriction is not operationally feasible under this model.
- How the assignment's cost and descriptive performance metrics relate to one another. Stability and lashing are reported metrics, not simultaneous objectives in the main run.
- How changes in costs or capacities affect the selected assignment and its feasibility.

### Concrete example

The data contain three OOG containers and four vessel slots, a cost and stability value for every container-slot pair, and weight limits for bays 2 and 3. The code uses these supplied values directly; it does not infer missing costs or constraints.

## Mathematical model and the Hungarian assignment

### Decision variable and basic assignment model

Let $C$ be the set of containers and $S$ the set of slots. The binary decision variable $x_{ij}$ records whether container $i$ is assigned to slot $j$: it is 1 when the pair is selected and 0 otherwise. Let $c_{ij}$ be the cost of that pair. The cost-based assignment problem is

$$\begin{aligned}
\min_{x}\quad & \sum_{i\in C}\sum_{j\in S} c_{ij}x_{ij} \\
\text{subject to}\quad
 & \sum_{j\in S}x_{ij}=1 && \forall i\in C,\\
 & \sum_{i\in C}x_{ij}\leq 1 && \forall j\in S,\\
 & x_{ij}\in\{0,1\} && \forall i\in C,\ j\in S.
\end{aligned}$$

The first constraint assigns every container exactly once. The second prevents a slot from receiving multiple containers. The binary condition makes each assignment a yes-or-no decision. In the code, $C$ and $S$ correspond to the rows and columns of `cost_matrix`, $c_{ij}$ to `cost_matrix[i, j]`, and $x_{ij}$ to an entry in the returned `assignment` matrix.

The Hungarian algorithm finds a minimum-cost solution to this basic assignment model. With three containers and four slots, each container is assigned and one slot remains unused. The assignment matrix is rectangular, with one row per container and one column per slot.

### Operational restrictions checked after assignment

Let $w_i$ and $h_i$ be the weight and height of container $i$, and let $W_j$ and $H_j$ be the weight capacity and height limit of slot $j$. Let $b(j)$ denote the bay containing slot $j$, and let $B_b$ be the maximum weight for bay $b$. An assignment passes the individual slot checks when every selected pair satisfies

$$x_{ij}=1\ \Longrightarrow\ w_i\leq W_j\ \text{and}\ h_i\leq H_j,$$

and it passes the bay check when

$$\sum_{i\in C}\sum_{j\in S:\,b(j)=b} w_i x_{ij}\leq B_b\qquad \forall b.$$

These are important feasibility conditions, but this notebook's Hungarian call optimizes the basic assignment model above and does not include these side constraints in its optimization. `verify_constraints` checks them on the returned assignment. If an audit fails, the assignment must not be described as feasible; the post-assignment audit alone does not search for a different constrained solution.

### Additional performance measures

For a selected assignment, the reported total stability is $\sum_{i,j}s_{ij}x_{ij}$, where $s_{ij}$ is the supplied stability score. The lashing-complexity score is $\sum_{i,j}\ell_i d_j x_{ij}$, where $\ell_i$ is container $i$'s lashing coefficient and $d_j$ is slot $j$'s crane distance. Capacity utilization is the weight assigned to selected slots divided by the sum of those slots' weight capacities, expressed as a percentage. These measures describe an assignment; they are not included in the main cost objective.

For a square assignment problem with $n$ rows and columns, the Hungarian algorithm has cubic time complexity, $O(n^3)$. The algorithm is polynomial-time and also supports rectangular assignment matrices. The audit and metric calculations in this small example require only loops over container-slot pairs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from dataclasses import dataclass
from typing import Dict, List, Tuple

# A consistent plotting style makes the assignment and capacity comparisons easier to read.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully.")

In [ ]:
@dataclass
class OOGContainer:
    """Store the attributes used to evaluate an OOG container assignment."""

    id: int
    weight: float  # tons
    height: float  # meters
    length: float  # meters
    container_type: str
    lashing_coeff: float


@dataclass
class VesselSlot:
    """Store slot limits and its bay and crane-distance information."""

    id: int
    weight_capacity: float  # tons
    height_limit: float  # meters
    bay_id: int
    crane_distance: float  # meters


@dataclass
class AssignmentProblem:
    """Collect the data needed to solve and audit an assignment instance."""

    containers: List[OOGContainer]
    slots: List[VesselSlot]
    cost_matrix: np.ndarray
    stability_matrix: np.ndarray
    bay_weight_limits: Dict[int, float]


print("Data structures defined.")

In [ ]:
# Container attributes are used in the slot and bay feasibility checks and metrics.
containers = [
    OOGContainer(1, 45, 2.8, 12.0, "flat rack", 2.1),
    OOGContainer(2, 32, 4.2, 6.0, "open-top", 3.8),
    OOGContainer(3, 38, 2.6, 12.0, "platform", 1.9),
]

# Slot attributes describe individual limits and bay-level accounting.
slots = [
    VesselSlot(1, 50, 3.0, 2, 15.0),
    VesselSlot(2, 40, 5.0, 3, 12.0),
    VesselSlot(3, 45, 3.0, 3, 18.0),
    VesselSlot(4, 60, 4.0, 3, 10.0),
]

# Rows correspond to containers and columns correspond to slots.
cost_matrix = np.array(
    [
        [100, 150, 120, 90],
        [80, 70, 110, 85],
        [95, 140, 100, 80],
    ]
)

# Larger stability scores indicate more favorable stability ratings.
stability_matrix = np.array(
    [
        [0.8, 0.7, 0.9, 0.85],
        [0.75, 0.8, 0.7, 0.9],
        [0.85, 0.65, 0.8, 0.75],
    ]
)

bay_weight_limits = {2: 80, 3: 120}  # tons per bay

problem = AssignmentProblem(
    containers,
    slots,
    cost_matrix,
    stability_matrix,
    bay_weight_limits,
)

assert cost_matrix.shape == (len(containers), len(slots))
assert stability_matrix.shape == cost_matrix.shape
assert len(containers) <= len(slots), "Every container needs an available slot."

print(f"Problem defined: {len(containers)} containers and {len(slots)} slots.")
print(f"Cost matrix shape: {cost_matrix.shape}")
print(f"Bay weight limits (tons): {bay_weight_limits}")

In [ ]:
def solve_hungarian_algorithm(cost_matrix: np.ndarray) -> Tuple[np.ndarray, float]:
    """Return the minimum-cost rectangular assignment and its total cost.

    The rows represent containers and the columns represent slots. This
    routine solves the basic assignment problem; side constraints such as
    weight, height, and bay limits are checked separately.
    """
    cost_matrix = np.asarray(cost_matrix)
    if cost_matrix.ndim != 2:
        raise ValueError("The cost matrix must be two-dimensional.")
    if cost_matrix.shape[0] == 0 or cost_matrix.shape[1] == 0:
        raise ValueError("The cost matrix must have at least one row and column.")
    if cost_matrix.shape[0] > cost_matrix.shape[1]:
        raise ValueError("There must be at least as many slots as containers.")
    if not np.isfinite(cost_matrix).all():
        raise ValueError("The cost matrix must contain only finite values.")

    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    assignment = np.zeros(cost_matrix.shape, dtype=int)
    assignment[row_ind, col_ind] = 1
    total_cost = float(cost_matrix[row_ind, col_ind].sum())

    return assignment, total_cost


def verify_constraints(
    problem: AssignmentProblem, assignment: np.ndarray
) -> Dict[str, bool]:
    """Audit assignment structure, slot limits, and bay-weight limits."""
    assignment = np.asarray(assignment)
    expected_shape = (len(problem.containers), len(problem.slots))
    if assignment.shape != expected_shape:
        raise ValueError(f"Assignment must have shape {expected_shape}.")
    if not np.isin(assignment, [0, 1]).all():
        raise ValueError("Assignment entries must be binary (0 or 1).")

    constraints = {}
    constraints["each_container_assigned"] = bool(
        np.all(assignment.sum(axis=1) == 1)
    )
    constraints["each_slot_max_once"] = bool(
        np.all(assignment.sum(axis=0) <= 1)
    )

    weight_capacity_satisfied = True
    height_constraints_satisfied = True
    bay_weights = {bay_id: 0.0 for bay_id in problem.bay_weight_limits}
    all_slot_bays_have_limits = all(
        slot.bay_id in problem.bay_weight_limits for slot in problem.slots
    )

    for i, container in enumerate(problem.containers):
        for j, slot in enumerate(problem.slots):
            if assignment[i, j] != 1:
                continue
            if container.weight > slot.weight_capacity:
                weight_capacity_satisfied = False
            if container.height > slot.height_limit:
                height_constraints_satisfied = False
            if slot.bay_id in bay_weights:
                bay_weights[slot.bay_id] += container.weight

    constraints["weight_capacity"] = weight_capacity_satisfied
    constraints["height_constraints"] = height_constraints_satisfied
    constraints["all_slot_bays_have_limits"] = all_slot_bays_have_limits
    constraints["bay_weight_limits"] = all_slot_bays_have_limits and all(
        bay_weights[bay_id] <= limit
        for bay_id, limit in problem.bay_weight_limits.items()
    )

    return constraints


print("Assignment and constraint-audit functions defined.")

In [ ]:
# The Hungarian algorithm minimizes the supplied cost matrix without side constraints.
assignment, total_cost = solve_hungarian_algorithm(problem.cost_matrix)

print("COST-MINIMIZING OOG CONTAINER ASSIGNMENT")
print("=" * 44)
print(f"Total assignment cost: {total_cost:g} cost units")

print("\nSelected container-slot pairs:")
for i, container in enumerate(problem.containers):
    for j, slot in enumerate(problem.slots):
        if assignment[i, j] == 1:
            print(
                f"  Container {container.id} ({container.container_type}, "
                f"{container.weight:g} tons, {container.height:g} m) -> "
                f"Slot {slot.id} (capacity {slot.weight_capacity:g} tons, "
                f"height limit {slot.height_limit:g} m, bay {slot.bay_id})"
            )
            print(
                f"    Cost: {problem.cost_matrix[i, j]:g} cost units; "
                f"stability score: {problem.stability_matrix[i, j]:.2f}"
            )

constraints = verify_constraints(problem, assignment)
print("\nConstraint audit:")
for constraint, satisfied in constraints.items():
    status = "PASS" if satisfied else "FAIL"
    print(f"  {constraint.replace('_', ' ').title()}: {status}")

all_satisfied = all(constraints.values())
print(f"\nOperational feasibility under the audited restrictions: {all_satisfied}")
assert np.isclose(
    total_cost,
    problem.cost_matrix[assignment == 1].sum(),
), "Reported assignment cost must equal the selected matrix entries."

In [ ]:
def calculate_performance_metrics(
    problem: AssignmentProblem, assignment: np.ndarray
) -> Dict[str, object]:
    """Calculate cost, stability, capacity utilization, lashing, and bay weights."""
    assignment = np.asarray(assignment)
    expected_shape = (len(problem.containers), len(problem.slots))
    if assignment.shape != expected_shape:
        raise ValueError(f"Assignment must have shape {expected_shape}.")
    if not np.isin(assignment, [0, 1]).all():
        raise ValueError("Assignment entries must be binary (0 or 1).")

    metrics = {}
    metrics["total_cost"] = float(
        problem.cost_matrix[assignment == 1].sum()
    )
    metrics["total_stability"] = float(
        problem.stability_matrix[assignment == 1].sum()
    )

    total_capacity_used = 0.0
    total_capacity_available = 0.0
    total_lashing_score = 0.0
    bay_weights = {bay_id: 0.0 for bay_id in problem.bay_weight_limits}

    for i, container in enumerate(problem.containers):
        for j, slot in enumerate(problem.slots):
            if assignment[i, j] != 1:
                continue
            total_capacity_used += container.weight
            total_capacity_available += slot.weight_capacity
            total_lashing_score += container.lashing_coeff * slot.crane_distance
            if slot.bay_id in bay_weights:
                bay_weights[slot.bay_id] += container.weight

    if total_capacity_available == 0:
        raise ValueError("Capacity utilization is undefined for an empty assignment.")

    metrics["capacity_utilization"] = (
        total_capacity_used / total_capacity_available
    ) * 100
    metrics["lashing_complexity"] = total_lashing_score
    metrics["bay_weights"] = bay_weights

    return metrics


metrics = calculate_performance_metrics(problem, assignment)

print("PERFORMANCE SUMMARY")
print("=" * 22)
print(f"Total cost: {metrics['total_cost']:g} cost units")
print(f"Total stability score: {metrics['total_stability']:.2f}")
print(f"Capacity utilization: {metrics['capacity_utilization']:.1f}%")
print(f"Lashing-complexity score: {metrics['lashing_complexity']:.1f}")

print("\nBay weight distribution:")
for bay_id, weight in metrics["bay_weights"].items():
    limit = problem.bay_weight_limits[bay_id]
    utilization = (weight / limit) * 100
    print(
        f"  Bay {bay_id}: {weight:g} of {limit:g} tons "
        f"({utilization:.1f}% of its limit)"
    )

In [ ]:
# The panels show the selected assignment alongside the slot and bay quantities used in the audit.
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    "OOG Container Assignment: Cost and Capacity Analysis",
    fontsize=16,
    fontweight="bold",
)

# Cost matrix: outlined cells identify the pairs selected by the Hungarian algorithm.
sns.heatmap(
    problem.cost_matrix,
    annot=True,
    fmt="g",
    cmap="RdYlBu_r",
    ax=ax1,
    xticklabels=[f"Slot {slot.id}" for slot in problem.slots],
    yticklabels=[f"Container {container.id}" for container in problem.containers],
)
ax1.set_title("Assignment Costs (cost units)")
ax1.set_xlabel("Vessel slots")
ax1.set_ylabel("OOG containers")
for i in range(len(problem.containers)):
    for j in range(len(problem.slots)):
        if assignment[i, j] == 1:
            ax1.add_patch(
                plt.Rectangle(
                    (j, i), 1, 1, fill=False, edgecolor="green", linewidth=3
                )
            )

# Container weights and heights use separate axes because their units differ.
container_names = [
    f"C{container.id}\n({container.container_type})"
    for container in problem.containers
]
weights = [container.weight for container in problem.containers]
heights = [container.height for container in problem.containers]
x_pos = np.arange(len(container_names))
bar_width = 0.35

ax2.bar(
    x_pos - bar_width / 2,
    weights,
    bar_width,
    label="Weight (tons)",
    alpha=0.8,
    color="skyblue",
)
ax2_twin = ax2.twinx()
ax2_twin.bar(
    x_pos + bar_width / 2,
    heights,
    bar_width,
    label="Height (m)",
    alpha=0.8,
    color="lightcoral",
)
ax2.set_xlabel("Containers")
ax2.set_ylabel("Weight (tons)")
ax2_twin.set_ylabel("Height (m)")
ax2.set_title("Container Characteristics")
ax2.set_xticks(x_pos)
ax2.set_xticklabels(container_names)
ax2.legend(loc="upper left")
ax2_twin.legend(loc="upper right")

# Slot bars compare the assigned container weight with each slot's individual capacity.
slot_names = [f"Slot {slot.id}\n(Bay {slot.bay_id})" for slot in problem.slots]
capacities = [slot.weight_capacity for slot in problem.slots]
used_capacity = np.zeros(len(problem.slots))
for i, container in enumerate(problem.containers):
    for j, slot in enumerate(problem.slots):
        if assignment[i, j] == 1:
            used_capacity[j] = container.weight

x_pos = np.arange(len(slot_names))
ax3.bar(
    x_pos,
    capacities,
    alpha=0.3,
    label="Available capacity",
    color="gray",
)
ax3.bar(
    x_pos,
    used_capacity,
    alpha=0.8,
    label="Assigned weight",
    color="orange",
)
ax3.set_xlabel("Vessel slots")
ax3.set_ylabel("Weight (tons)")
ax3.set_title("Slot Weight Capacity")
ax3.set_xticks(x_pos)
ax3.set_xticklabels(slot_names)
ax3.legend()
for i, (used, capacity) in enumerate(zip(used_capacity, capacities)):
    if used > 0:
        ax3.text(
            i,
            used + 1,
            f"{used / capacity * 100:.1f}%",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

# Bay totals are compared with the explicit bay limits.
bay_ids = list(problem.bay_weight_limits)
bay_limits = [problem.bay_weight_limits[bay_id] for bay_id in bay_ids]
bay_weights_actual = [metrics["bay_weights"][bay_id] for bay_id in bay_ids]
x_pos = np.arange(len(bay_ids))
bar_width = 0.35
ax4.bar(
    x_pos - bar_width / 2,
    bay_limits,
    bar_width,
    label="Weight limit",
    alpha=0.3,
    color="red",
)
ax4.bar(
    x_pos + bar_width / 2,
    bay_weights_actual,
    bar_width,
    label="Assigned weight",
    alpha=0.8,
    color="green",
)
ax4.set_xlabel("Bay")
ax4.set_ylabel("Weight (tons)")
ax4.set_title("Bay Weight Limits")
ax4.set_xticks(x_pos)
ax4.set_xticklabels([f"Bay {bay_id}" for bay_id in bay_ids])
ax4.legend()
for i, (actual, limit) in enumerate(zip(bay_weights_actual, bay_limits)):
    ax4.text(
        i,
        actual + 1,
        f"{actual / limit * 100:.1f}%",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

plt.tight_layout()
plt.show()
print("Assignment and capacity visualizations created.")

In [ ]:
print("SENSITIVITY ANALYSIS: WHAT-IF SCENARIOS")
print("=" * 45)

# Scenario 1 changes the cost of assigning any container to Slot 4.
modified_cost_matrix_1 = problem.cost_matrix.copy()
modified_cost_matrix_1[:, 3] += 50
assignment_1, cost_1 = solve_hungarian_algorithm(modified_cost_matrix_1)
cost_change = cost_1 - total_cost
cost_change_percent = (cost_change / total_cost) * 100 if total_cost else 0.0

print("\nScenario 1: Slot 4 assignment costs increase by 50 cost units")
print(f"  New minimum assignment cost: {cost_1:g} cost units")
print(f"  Change from the baseline cost: {cost_change:g} cost units "
      f"({cost_change_percent:.1f}%)")
print("  Selected pairs:")
for i, container in enumerate(problem.containers):
    for j, slot in enumerate(problem.slots):
        if assignment_1[i, j] == 1:
            print(
                f"    Container {container.id} -> Slot {slot.id} "
                f"(modified cost: {modified_cost_matrix_1[i, j]:g})"
            )
scenario_1_constraints = verify_constraints(problem, assignment_1)
print(
    "  Passes the audited restrictions: "
    f"{all(scenario_1_constraints.values())}"
)

# Scenario 2 changes Slot 2's weight capacity and audits the baseline assignment.
modified_slots = problem.slots.copy()
modified_slots[1] = VesselSlot(2, 30, 5.0, 3, 12.0)
modified_problem = AssignmentProblem(
    problem.containers,
    modified_slots,
    problem.cost_matrix,
    problem.stability_matrix,
    problem.bay_weight_limits,
)
scenario_2_constraints = verify_constraints(modified_problem, assignment)

print("\nScenario 2: Slot 2 weight capacity decreases from 40 to 30 tons")
print("  Audit of the baseline assignment:")
for constraint, satisfied in scenario_2_constraints.items():
    print(
        f"    {constraint.replace('_', ' ').title()}: "
        f"{'PASS' if satisfied else 'FAIL'}"
    )
print(
    "  Baseline assignment remains feasible: "
    f"{all(scenario_2_constraints.values())}"
)
print(
  "  This scenario audits the baseline assignment; it does not rerun "
  "the Hungarian algorithm with capacity restrictions."
)

# Scenario 3 uses a transformed cost matrix to maximize total stability.
# Since every container is assigned, minimizing 1 - stability is equivalent
# to maximizing the sum of selected stability scores.
stability_cost_matrix = 1 - problem.stability_matrix
assignment_3, transformed_cost = solve_hungarian_algorithm(
    stability_cost_matrix
)
actual_stability = float(problem.stability_matrix[assignment_3 == 1].sum())
actual_cost = float(problem.cost_matrix[assignment_3 == 1].sum())
scenario_3_constraints = verify_constraints(problem, assignment_3)
stability_change = actual_stability - metrics["total_stability"]
cost_change_from_stability = actual_cost - total_cost

print("\nScenario 3: Maximize the sum of stability scores")
print(f"  Stability score: {actual_stability:.2f}")
print(f"  Baseline cost-minimizing assignment stability: "
      f"{metrics['total_stability']:.2f}")
print(f"  Stability-score change: {stability_change:+.2f}")
print(f"  Assignment cost: {actual_cost:g} cost units")
print(f"  Baseline assignment cost: {total_cost:g} cost units")
print(f"  Assignment-cost change: {cost_change_from_stability:+g} cost units")
print("  Selected pairs:")
for i, container in enumerate(problem.containers):
    for j, slot in enumerate(problem.slots):
        if assignment_3[i, j] == 1:
            print(
                f"    Container {container.id} -> Slot {slot.id} "
                f"(stability: {problem.stability_matrix[i, j]:.2f}, "
                f"cost: {problem.cost_matrix[i, j]:g})"
            )
print("  Constraint audit:")
for constraint, satisfied in scenario_3_constraints.items():
    print(
        f"    {constraint.replace('_', ' ').title()}: "
        f"{'PASS' if satisfied else 'FAIL'}"
    )
print(
    "  Stability-maximizing assignment passes all audited restrictions: "
    f"{all(scenario_3_constraints.values())}"
)

print("\nInterpretation:")
print("  Cost changes can alter the minimum-cost assignment.")
print("  A reduced slot capacity can make an existing assignment infeasible.")
print(
    "  Optimizing stability instead of cost can produce a different "
    "assignment; its feasibility must be checked separately."
)

## Interpreting the method and its results

The main run uses `solve_hungarian_algorithm` to minimize the sum of the supplied container-slot costs. The returned assignment is a solution to the basic one-to-one assignment problem. `verify_constraints` then reports whether that assignment also meets the modeled weight, height, and bay restrictions. Read these two results together: a low-cost assignment that fails an audit is not a feasible operational plan under the stated assumptions.

`calculate_performance_metrics` reports stability, capacity utilization, lashing complexity, and bay weights for the selected assignment. These quantities help describe the plan, but only cost is optimized in the main run. Scenario 3 changes the assignment objective to stability for that separate run; the feasibility audit remains necessary because the Hungarian algorithm still does not enforce the side constraints.

The method provides a polynomial-time solution to the basic assignment problem and a transparent way to inspect its output. The post-assignment checks are essential, but they do not turn the basic Hungarian assignment into an optimizer for the full set of side-constrained decisions. In applications where the cost-minimizing assignment fails a restriction, this tier identifies the infeasibility rather than claiming to have found the least-cost feasible plan.

### Appropriate use and limitations

This formulation is useful when assignment costs are known, each container needs one distinct slot, and the assignment structure is the main optimization problem. It also provides clearly defined metrics and explicit feasibility checks for the example.

The results assume fixed input data and do not account for uncertainty, operational changes, or interactions between containers beyond the stated bay-weight totals. The stability and lashing quantities are descriptive measures unless explicitly selected as the assignment objective. The supplied sensitivity scenarios illustrate changes in the inputs or objective; they do not add side constraints to the Hungarian optimization.